# Dependencies
#### Run the following file(s) before running this code.
- 03_baseline_similarity_graph.ipynb (or 03b or 03c)
- 07_gds_Louvain_Summary.ipynb

##### Note:
URL of the Neo4j browser:
- https://[IP address]:7473/browser/

ID & Pass: 
- Use the one in .env


In [1]:
# Config
SAMPLING:bool       = True
NUM_SAMPLE:int      = 250   # Number of sample data to be ingested to the graph database
SUMMARY_SAMPLE:int  = 20    # Number of samples as inputs of summarizing
RAND_SEED:int       = 77    # Seed for sampling
NUM_SIM:int         = 5     # Number of results from KNN search (does not include the own node)
EMBEDDING_MODEL:str = "text-embedding-3-small"
MAX_TOKENS:int      = 7800  # Max 8192 - some safety buffer about 5%
INDEX_NAME:str      = "idx:complaints_vss"
FILE_PATH:str       = "../../data/original/complaints-2025-11-02_04_18.csv"

In [2]:
import time
from datetime import datetime, timedelta

In [3]:
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display
import tiktoken

In [4]:
from dotenv import load_dotenv  
load_dotenv()

True

In [5]:
import neo4j

In [6]:
# Ignore unclosed SSL socket warnings - optional in case you get these errors
import warnings

In [7]:
warnings.filterwarnings(action="ignore", message="unclosed", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning) 

In [8]:
# Show all columns
pd.set_option('display.max_columns', None)

# Show all rows
pd.set_option('display.max_rows', None)

In [9]:
# Timestamp (Start)
current_datetime = datetime.now()
formatted_time = current_datetime.strftime("%Y-%m-%d_%H:%M:%S")
print(formatted_time)

start_time = time.time()

2026-01-08_22:23:15


### Neo4j

In [10]:
driver = neo4j.GraphDatabase.driver(
    uri=os.environ.get("NEO4J_URI"), 
    auth=(os.environ.get("NEO4J_USERNAME"), 
          os.environ.get("NEO4J_PASSWORD"))
)

In [11]:
session = driver.session(database="neo4j")

In [12]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

# Write in-memory graph to the actual knowledge graph (Materialized)

In [13]:
query = """
MATCH (ca:Category)<-[:IN_CATEGORY]-(c1:Complaint)
    -[s:SIMILAR]- 
    (c2:Complaint)-[:IN_CATEGORY]->(cb:Category)

WHERE elementId(ca) < elementId(cb)   // avoid double counting and self-loops

// Note:
// - similarity: Average similarity score
// - pairCount: Number of original relationships (based on Complaints level)
WITH 
    ca, cb,
    avg(toFloat(s.similarity_score)) AS similarity,
    count(*) AS pairCount

MERGE (ca)-[r:CATEGORY_SIMILAR_TO]->(cb)
SET r.similarity = similarity,
    r.pairCount = pairCount;

"""

session.run(query)

# Degree Centrality

In [14]:
# Drop the in-memory graph
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

# Create an in-memory graph named 'ds_graph'
# Only include 'Category' nodes and 'CATEGORY_SIMILAR_TO' relationship
query = """
CALL gds.graph.project(
    'ds_graph', 
    'Category', 
    'CATEGORY_SIMILAR_TO'
)
"""
session.run(query)

In [15]:
# Note: 'UNDIRECTED': Count both in and out going arrows
query = """
CALL gds.degree.stream(
    'ds_graph',
    {orientation: 'UNDIRECTED'}
)
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).summary AS Category, score AS Degree
ORDER BY Degree DESC, Category;
"""

my_neo4j_run_query_pandas(query).head(10)
# session.run(query)

,Category,Degree
0,Identity theft and fraudulent credit activity.,8.0
1,Fraudulent accounts resulting from data breach...,7.0
2,Failure to correct inaccuracies in credit repo...,6.0
3,Inaccurate credit reporting and failure to val...,6.0
4,Unauthorized accounts and inaccuracies on cred...,6.0
5,Fraudulent accounts due to data breaches and u...,5.0
6,Violation of consumer privacy and inaccurate c...,5.0
7,Identity theft and its consequences.,3.0
8,Identity theft and unauthorized use of persona...,3.0
9,Unauthorized accounts and failure to investiga...,1.0


# Harmonic Centrality

In [16]:
# Drop the in-memory graph
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

# Create an in-memory graph named 'ds_graph'
# Only include 'Category' nodes and 'CATEGORY_SIMILAR_TO' relationship
query = """
CALL gds.graph.project(
    'ds_graph', 
    'Category', 
    'CATEGORY_SIMILAR_TO'
)
"""
session.run(query)

In [17]:
query = """

CALL gds.closeness.harmonic.stream('ds_graph', {})
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).summary AS Category, score AS Closeness
ORDER BY Closeness DESC, Category;

"""

my_neo4j_run_query_pandas(query).head(10)

# Note: 
# This is Harmonic Centrality based on Category nodes and Complaint nodes.
# Considering all Complaint nodes are directly connected to Category nodes (by the design of this graph),
# the Closeness score in here has almost the same meaning as the Centrality analysis above

,Category,Closeness
0,Unauthorized accounts and inaccuracies on cred...,0.777778
1,Failure to correct inaccuracies in credit repo...,0.611111
2,Inaccurate credit reporting and failure to val...,0.555556
3,Fraudulent accounts due to data breaches and u...,0.444444
4,Unauthorized accounts and failure to investiga...,0.444444
5,Fraudulent accounts resulting from data breach...,0.388889
6,Identity theft and its consequences.,0.277778
7,Identity theft and unauthorized use of persona...,0.166667
8,Identity theft and fraudulent credit activity.,0.111111
9,Violation of consumer privacy and inaccurate c...,0.000000


# PageRank analysis

In [18]:
# Drop the in-memory graph
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

# Create an in-memory graph named 'ds_graph'
# Only include 'Category' nodes and 'CATEGORY_SIMILAR_TO' relationship
query = """
CALL gds.graph.project(
    'ds_graph', 
    'Category', 
    'CATEGORY_SIMILAR_TO'
)
"""
session.run(query)

In [19]:
query = """

CALL gds.pageRank.stream('ds_graph',
                         { maxIterations: $max_iterations,
                           dampingFactor: $damping_factor}
                         )
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).summary AS Category, score as page_rank
ORDER BY page_rank DESC, Category ASC

"""

max_iterations = 20
damping_factor = 0.05

my_neo4j_run_query_pandas(query, max_iterations=max_iterations, damping_factor=damping_factor).head(10)

,Category,page_rank
0,Unauthorized accounts and inaccuracies on cred...,1.053908
1,Fraudulent accounts resulting from data breach...,1.029814
2,Failure to correct inaccuracies in credit repo...,1.004319
3,Inaccurate credit reporting and failure to val...,1.003707
4,Identity theft and its consequences.,0.980775
5,Fraudulent accounts due to data breaches and u...,0.979226
6,Unauthorized accounts and failure to investiga...,0.975108
7,Identity theft and fraudulent credit activity.,0.959500
8,Identity theft and unauthorized use of persona...,0.956854
9,Violation of consumer privacy and inaccurate c...,0.950000


In [20]:
# Timestamp (End)
current_datetime = datetime.now()
formatted_time = current_datetime.strftime("%Y-%m-%d_%H:%M:%S")
print(formatted_time)

end_time = time.time()
elapsed_seconds = end_time - start_time

# Convert elapsed seconds to minutes and seconds
minutes = int(elapsed_seconds // 60)
seconds = elapsed_seconds % 60

print(f"Program elapsed time: {minutes} minutes and {seconds:.2f} seconds")

2026-01-08_22:23:15
Program elapsed time: 0 minutes and 0.35 seconds
